# **Import libraries**

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor

In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
train_df=pd.read_csv('data/train_data.csv')

In [ ]:
test_df=pd.read_csv('data/test_data.csv')

In [ ]:
train_raw=train_df.copy()

# **Exploraity Data Analysis**

**Checks (shapes, columns, duplicates, missing values)**

Train Data

In [ ]:
train_df.head()

In [ ]:
train_df.value_counts().head(10)

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
train_df.duplicated().sum()

In [ ]:
train_df.nunique()

In [ ]:
train_df["Company"].value_counts()

In [ ]:
train_df["CPU_Type"].value_counts().head(20)

In [ ]:
train_df["CPU_Type"].value_counts().head(20)

In [ ]:
train_df.groupby('Company')['Price (Euro)'].mean()

In [ ]:
train_df.groupby('TypeName')['Price (Euro)'].mean()

In [ ]:
train_df.groupby('CPU_Company')['Price (Euro)'].mean()

In [ ]:
train_df.groupby('GPU_Company')['Price (Euro)'].mean()

In [ ]:
train_df.groupby('OpSys')['Price (Euro)'].mean()

#Visualisation

In [ ]:
for i in train_df.select_dtypes(include="number").columns:
  sns.boxplot(data=train_df, x=i)
  plt.show()

In [ ]:
for col in train_df.select_dtypes(include="number").columns:
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=train_df, x=col, y='Price (Euro)')
    plt.title(f'{col} vs Price')
    plt.show()

In [ ]:
numeric_df=train_df.select_dtypes(include="number")
numeric_df.head()
corr_df=numeric_df.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr_df, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=train_df,x="Company",order=train_df["Company"].value_counts().index)
plt.xticks(rotation=45)
plt.title("number of laptops per Company")
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
avg_price=train_df.groupby("Company")["Price (Euro)"].mean().sort_values()
avg_price.plot(kind="bar")
plt.title("Average price per Company")
plt.ylabel("Average price")
plt.xticks(rotation=45)
plt.show()

# **Data Preparation**

**Preprocessing**

In [ ]:
y = train_df["Price (Euro)"].copy()

test_ids = test_df["id"].copy()

X = train_df.drop(columns=["Price (Euro)"]).copy()

X = X.drop(columns=["id"])
test_features = test_df.drop(columns=["id"]).copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("test_features shape:", test_features.shape)

print("\nRemaining columns:")
print(X.columns.tolist())

**Feature Engineering**

Screen Resolution

In [ ]:
def screen_resolution_features(df):
    df = df.copy()

    df["is_ips"] = df["ScreenResolution"].str.contains("IPS", case=False, na=False).astype(int)
    df["is_touch"] = df["ScreenResolution"].str.contains("Touchscreen", case=False, na=False).astype(int)

    resolution = df["ScreenResolution"].str.extract(r"(\d{3,4})x(\d{3,4})")
    df["res_width"] = pd.to_numeric(resolution[0], errors="coerce")
    df["res_height"] = pd.to_numeric(resolution[1], errors="coerce")

    df["ppi"] = np.sqrt(df["res_width"]**2 + df["res_height"]**2) / df["Inches"]

    df = df.drop(columns=["ScreenResolution","Inches","res_height","res_width"])

    return df
X = screen_resolution_features(X)
test_features = screen_resolution_features(test_features)

print("\nMissing values after screen engineering:")

Calculating Memory

In [ ]:
def convert_to_gb(size, unit):
    size = float(size)
    if unit.upper() == "TB":
        return size * 1024
    return size

def parse_memory_column(df):
    df = df.copy()

    ssd_list = []
    hdd_list = []
    flash_list = []
    hybrid_list = []

    for value in df["Memory"]:
        ssd = 0
        hdd = 0
        flash = 0
        hybrid = 0

        text = str(value).upper().strip()
        parts = re.split(r"\s*\+\s*", text)

        for part in parts:
            match = re.search(r"(\d+(?:\.\d+)?)\s*(GB|TB)", part)
            if not match:
                continue

            size = convert_to_gb(match.group(1), match.group(2))

            if "SSD" in part:
                ssd += size
            elif "HDD" in part:
                hdd += size
            elif "HYBRID" in part:
                hybrid += size
            elif "FLASH STORAGE" in part or "EMMC" in part:
                flash += size
            else:
                # fallback for uncommon labels
                flash += size

        ssd_list.append(ssd)
        hdd_list.append(hdd)
        flash_list.append(flash)
        hybrid_list.append(hybrid)

    df["ssd_gb"] = ssd_list
    df["hdd_gb"] = hdd_list
    df["flash_gb"] = flash_list
    df["hybrid_gb"] = hybrid_list

    df["total_storage_gb"] = df["ssd_gb"] + df["hdd_gb"] + df["flash_gb"] + df["hybrid_gb"]

    df["has_ssd"] = (df["ssd_gb"] > 0).astype(int)
    df["has_hdd"] = (df["hdd_gb"] > 0).astype(int)
    df["has_flash"] = (df["flash_gb"] > 0).astype(int)
    df["has_hybrid"] = (df["hybrid_gb"] > 0).astype(int)

    df = df.drop(columns=["Memory"])
    return df

X = parse_memory_column(X)
test_features = parse_memory_column(test_features)

print(
    X[
        [
            "ssd_gb", "hdd_gb", "flash_gb", "hybrid_gb",
            "total_storage_gb", "has_ssd", "has_hdd", "has_flash", "has_hybrid"
        ]
    ].head(10)
)

print("\nMissing values after memory engineering:")
print(
    X[
        [
            "ssd_gb", "hdd_gb", "flash_gb", "hybrid_gb",
            "total_storage_gb"
        ]
    ].isnull().sum()
)

separating cpu type

In [ ]:
def extract_cpu_family(text):
    text = str(text).upper()

    if "CORE I7" in text:
        return "Core i7"
    elif "CORE I5" in text:
        return "Core i5"
    elif "CORE I3" in text:
        return "Core i3"
    elif "CORE M" in text:
        return "Core M"
    elif "CELERON" in text:
        return "Celeron"
    elif "PENTIUM" in text:
        return "Pentium"
    elif "RYZEN" in text:
        return "Ryzen"
    elif "XEON" in text:
        return "Xeon"
    elif "ATOM" in text:
        return "Atom"
    elif "A12-SERIES" in text:
        return "A12-Series"
    elif "A10-SERIES" in text:
        return "A10-Series"
    elif "A9-SERIES" in text:
        return "A9-Series"
    elif "A8-SERIES" in text:
        return "A8-Series"
    elif "A6-SERIES" in text:
        return "A6-Series"
    elif "E-SERIES" in text:
        return "E-Series"
    elif "FX" in text:
        return "FX"
    else:
        return "Other"

def extract_cpu_core_class(text):
    text = str(text).upper()
    if "DUAL CORE" in text:
        return "Dual"
    elif "QUAD CORE" in text:
        return "Quad"
    else:
        return "None"

def extract_cpu_suffix(text):
    text = str(text).upper()

    # Common laptop suffixes
    match = re.search(r"\d+(HQ|HK|U|Y|M|P|G)\b", text)
    if match:
        return match.group(1)

    # Some CPUs may contain suffix after hyphen
    match2 = re.search(r"\b(HQ|HK|U|Y|M|P|G)\b", text)
    if match2:
        return match2.group(1)

    return "None"

def extract_cpu_generation(text):
    text = str(text).upper()

    family = extract_cpu_family(text)

    # safest for Intel Core family
    if family in ["Core i3", "Core i5", "Core i7", "Core M"]:
        match = re.search(r"\b(\d{4,5})[A-Z]*\b", text)
        if match:
            num = match.group(1)

            # 7500U -> 7, 8250U -> 8, 6Y75 -> 6, 7Y30 -> 7
            if len(num) == 4:
                return num[0]
            elif len(num) == 5:
                return num[:2]

    return "Unknown"

def cpu_features(df):
    df = df.copy()

    df["cpu_family"] = df["CPU_Type"].apply(extract_cpu_family)
    df["cpu_core_class"] = df["CPU_Type"].apply(extract_cpu_core_class)
    df["cpu_suffix"] = df["CPU_Type"].apply(extract_cpu_suffix)
    df["cpu_generation"] = df["CPU_Type"].apply(extract_cpu_generation)

    df = df.drop(columns=["CPU_Type"])
    return df

X = cpu_features(X)
test_features = cpu_features(test_features)

print(X[["cpu_family", "cpu_core_class", "cpu_suffix", "cpu_generation"]].head(15))

print("\nTrain CPU family counts:")
print(X["cpu_family"].value_counts())

print("\nTrain CPU suffix counts:")
print(X["cpu_suffix"].value_counts())

print("\nTrain CPU generation counts:")
print(X["cpu_generation"].value_counts())

product

In [ ]:
def extract_product_series(text):
    text = str(text).strip()
    if text == "" or text.lower() == "nan":
        return "Unknown"
    return text.split()[0]

def extract_gpu_series(text):
    text = str(text).upper()

    if "GTX" in text:
        return "GTX"
    elif "RTX" in text:
        return "RTX"
    elif "MX" in text:
        return "MX"
    elif "QUADRO" in text:
        return "Quadro"
    elif "IRIS" in text:
        return "Iris"
    elif "UHD" in text:
        return "UHD"
    elif "HD GRAPHICS" in text or re.search(r"\bHD\b", text):
        return "HD Graphics"
    elif "RADEON" in text:
        return "Radeon"
    elif "RX" in text:
        return "RX"
    elif "FIREPRO" in text:
        return "FirePro"
    elif "GEFORCE" in text:
        return "GeForce"
    else:
        return "Other"

def gpu_product_features(df):
    df = df.copy()

    df["product_series"] = df["Product"].apply(extract_product_series)
    df["gpu_series"] = df["GPU_Type"].apply(extract_gpu_series)

    df = df.drop(columns=["Product", "GPU_Type"])
    return df

X = gpu_product_features(X)
test_features = gpu_product_features(test_features)

print(X[["product_series", "gpu_series"]].head(15))

print("\nTop product series:")
print(X["product_series"].value_counts().head(20))

print("\nGPU series counts:")
print(X["gpu_series"].value_counts())

In [ ]:
def clean_text_categories(df):
    df = df.copy()

    product_map = {
        "Macbook": "MacBook",
        "Thinkpad": "ThinkPad",
        "Probook": "ProBook",
        "Rog": "ROG",
        "Vivobook": "VivoBook"
    }
    df["product_series"] = df["product_series"].replace(product_map)

    df["OpSys"] = df["OpSys"].replace({
        "Mac OS X": "macOS",
        "macOS": "macOS"
    })

    return df
X = clean_text_categories(X)
test_features = clean_text_categories(test_features)

print("Top product series after cleaning:")

print("\nUnique values in OpSys:")
print(X["OpSys"].value_counts())

In [ ]:
print("X shape:", X.shape)
print("test_features shape:", test_features.shape)
print("y shape:", y.shape)

print("\nColumns in X:")
print(X.columns.tolist())

print("\nMissing values in X:")
print(X.isnull().sum()[X.isnull().sum() > 0])

print("\nMissing values in test_features:")
print(test_features.isnull().sum()[test_features.isnull().sum() > 0])

print("\nColumns mismatch check:")
print("Only in X:", set(X.columns) - set(test_features.columns))
print("Only in test_features:", set(test_features.columns) - set(X.columns))

display(X.head())

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nNumber of categorical columns:", len(categorical_cols))

print("\nNumerical columns:")
print(numerical_cols)

print("\nNumber of numerical columns:", len(numerical_cols))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Preprocessor for linear-style models
linear_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

# Preprocessor for tree model
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

print("Linear preprocessor created.")
print("Tree preprocessor created.")

In [ ]:
from sklearn.model_selection import train_test_split

# Holdout split
X_train_main, X_val, y_train_main, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

# Log-transform target for experiments
y_train_main_log = np.log1p(y_train_main)
y_val_log = np.log1p(y_val)

print("X_train_main shape:", X_train_main.shape)
print("X_val shape:", X_val.shape)
print("y_train_main shape:", y_train_main.shape)
print("y_val shape:", y_val.shape)

print("\nOriginal target range:")
print("min =", y.min(), ", max =", y.max())

print("\nLog target range:")
print("min =", y_train_main_log.min(), ", max =", y_train_main_log.max())

model

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

linear_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", LinearRegression())
])

linear_cv_scores = cross_val_score(
    linear_model,
    X_train_main,
    y_train_main,
    cv=kf,
    scoring="r2"
)

print("Linear Regression CV R² scores:", linear_cv_scores)
print("Mean CV R²:", linear_cv_scores.mean())
print("Std CV R²:", linear_cv_scores.std())

In [ ]:
linear_model.fit(X_train_main, y_train_main)

y_val_pred_linear = linear_model.predict(X_val)

linear_val_r2 = r2_score(y_val, y_val_pred_linear)
linear_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_linear))
linear_val_mae = mean_absolute_error(y_val, y_val_pred_linear)

print("Linear Regression Validation R²:", linear_val_r2)
print("Linear Regression Validation RMSE:", linear_val_rmse)
print("Linear Regression Validation MAE:", linear_val_mae)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred_linear, alpha=0.7)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Linear Regression: Actual vs Predicted on Validation Set")
plt.show()

In [ ]:
linear_residuals = y_val - y_val_pred_linear

plt.figure(figsize=(8, 6))
plt.scatter(y_val_pred_linear, linear_residuals, alpha=0.7)
plt.axhline(y=0)
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Linear Regression: Residual Plot")
plt.show()

In [ ]:


train_sizes, train_scores, val_scores = learning_curve(
    estimator=linear_model,
    X=X_train_main,
    y=y_train_main,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

train_mse = -train_scores
val_mse = -val_scores

train_mse_mean = train_mse.mean(axis=1)
train_mse_std = train_mse.std(axis=1)

val_mse_mean = val_mse.mean(axis=1)
val_mse_std = val_mse.std(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mse_mean, label="Training MSE")
plt.plot(train_sizes, val_mse_mean, label="Validation MSE")

plt.fill_between(
    train_sizes,
    train_mse_mean - train_mse_std,
    train_mse_mean + train_mse_std,
    alpha=0.2
)

plt.fill_between(
    train_sizes,
    val_mse_mean - val_mse_std,
    val_mse_mean + val_mse_std,
    alpha=0.2
)

plt.xlabel("Training Set Size")
plt.ylabel("MSE")
plt.title("Linear Regression Learning Curve")
plt.legend()
plt.show()

In [ ]:
linear_results = {
    "Model": "Linear Regression",
    "CV_R2": linear_cv_scores.mean(),
    "Validation_R2": linear_val_r2,
    "Validation_RMSE": linear_val_rmse,
    "Validation_MAE": linear_val_mae
}
linear_results

ridge model

In [ ]:
ridge_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", Ridge(alpha=1.0))
])

ridge_cv_scores = cross_val_score(
    ridge_model,
    X_train_main,
    y_train_main,
    cv=kf,
    scoring="r2"
)

print("Ridge Regression CV R² scores:", ridge_cv_scores)
print("Mean CV R²:", ridge_cv_scores.mean())
print("Std CV R²:", ridge_cv_scores.std())

In [ ]:
ridge_model.fit(X_train_main, y_train_main)

y_val_pred_ridge_baseline = ridge_model.predict(X_val)

ridge_baseline_val_r2 = r2_score(y_val, y_val_pred_ridge_baseline)
ridge_baseline_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_ridge_baseline))
ridge_baseline_val_mae = mean_absolute_error(y_val, y_val_pred_ridge_baseline)

print("Ridge Regression Validation R²:", ridge_baseline_val_r2)
print("Ridge Regression Validation RMSE:", ridge_baseline_val_rmse)
print("Ridge Regression Validation MAE:", ridge_baseline_val_mae)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred_ridge_baseline, alpha=0.7)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Ridge Regression (Baseline): Actual vs Predicted on Validation Set")
plt.show()

In [ ]:
train_sizes_ridge, train_scores_ridge, val_scores_ridge = learning_curve(
    estimator=ridge_model,
    X=X_train_main,
    y=y_train_main,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

train_mse_ridge = -train_scores_ridge
val_mse_ridge = -val_scores_ridge

train_mse_ridge_mean = train_mse_ridge.mean(axis=1)
train_mse_ridge_std = train_mse_ridge.std(axis=1)

val_mse_ridge_mean = val_mse_ridge.mean(axis=1)
val_mse_ridge_std = val_mse_ridge.std(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes_ridge, train_mse_ridge_mean, label="Training MSE")
plt.plot(train_sizes_ridge, val_mse_ridge_mean, label="Validation MSE")

plt.fill_between(
    train_sizes_ridge,
    train_mse_ridge_mean - train_mse_ridge_std,
    train_mse_ridge_mean + train_mse_ridge_std,
    alpha=0.2
)

plt.fill_between(
    train_sizes_ridge,
    val_mse_ridge_mean - val_mse_ridge_std,
    val_mse_ridge_mean + val_mse_ridge_std,
    alpha=0.2
)

plt.xlabel("Training Set Size")
plt.ylabel("MSE")
plt.title("Ridge Regression (Baseline) Learning Curve")
plt.legend()
plt.show()

In [ ]:
ridge_baseline_results = {
    "Model": "Ridge (baseline)",
    "CV_R2": ridge_cv_scores.mean(),
    "Validation_R2": ridge_baseline_val_r2,
    "Validation_RMSE": ridge_baseline_val_rmse,
    "Validation_MAE": ridge_baseline_val_mae
}

ridge_baseline_results

lasso

In [ ]:
lasso_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", Lasso(alpha=0.0005, max_iter=50000))
])

lasso_cv_scores = cross_val_score(
    lasso_model,
    X_train_main,
    y_train_main,
    cv=kf,
    scoring="r2"
)

print("Lasso Regression CV R² scores:", lasso_cv_scores)
print("Mean CV R²:", lasso_cv_scores.mean())
print("Std CV R²:", lasso_cv_scores.std())

In [ ]:
lasso_model.fit(X_train_main, y_train_main)

y_val_pred_lasso = lasso_model.predict(X_val)

lasso_val_r2 = r2_score(y_val, y_val_pred_lasso)
lasso_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_lasso))
lasso_val_mae = mean_absolute_error(y_val, y_val_pred_lasso)

print("Lasso Regression Validation R²:", lasso_val_r2)
print("Lasso Regression Validation RMSE:", lasso_val_rmse)
print("Lasso Regression Validation MAE:", lasso_val_mae)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred_lasso, alpha=0.7)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Lasso Regression: Actual vs Predicted on Validation Set")
plt.show()

In [ ]:
lasso_results = {
    "Model": "Lasso",
    "CV_R2": lasso_cv_scores.mean(),
    "Validation_R2": lasso_val_r2,
    "Validation_RMSE": lasso_val_rmse,
    "Validation_MAE": lasso_val_mae
}

lasso_results

Tree

In [ ]:
tree_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))
])

tree_cv_scores = cross_val_score(
    tree_model,
    X_train_main,
    y_train_main,
    cv=kf,
    scoring="r2"
)

print("Decision Tree CV R² scores:", tree_cv_scores)
print("Mean CV R²:", tree_cv_scores.mean())
print("Std CV R²:", tree_cv_scores.std())

In [ ]:
tree_model.fit(X_train_main, y_train_main)

y_val_pred_tree_baseline = tree_model.predict(X_val)

tree_baseline_val_r2 = r2_score(y_val, y_val_pred_tree_baseline)
tree_baseline_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_tree_baseline))
tree_baseline_val_mae = mean_absolute_error(y_val, y_val_pred_tree_baseline)

print("Decision Tree Validation R²:", tree_baseline_val_r2)
print("Decision Tree Validation RMSE:", tree_baseline_val_rmse)
print("Decision Tree Validation MAE:", tree_baseline_val_mae)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred_tree_baseline, alpha=0.7)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Decision Tree (Baseline): Actual vs Predicted on Validation Set")
plt.show()

In [ ]:
train_sizes_tree, train_scores_tree, val_scores_tree = learning_curve(
    estimator=tree_model,
    X=X_train_main,
    y=y_train_main,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=kf,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

train_mse_tree = -train_scores_tree
val_mse_tree = -val_scores_tree

train_mse_tree_mean = train_mse_tree.mean(axis=1)
train_mse_tree_std = train_mse_tree.std(axis=1)

val_mse_tree_mean = val_mse_tree.mean(axis=1)
val_mse_tree_std = val_mse_tree.std(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes_tree, train_mse_tree_mean, label="Training MSE")
plt.plot(train_sizes_tree, val_mse_tree_mean, label="Validation MSE")

plt.fill_between(
    train_sizes_tree,
    train_mse_tree_mean - train_mse_tree_std,
    train_mse_tree_mean + train_mse_tree_std,
    alpha=0.2
)

plt.fill_between(
    train_sizes_tree,
    val_mse_tree_mean - val_mse_tree_std,
    val_mse_tree_mean + val_mse_tree_std,
    alpha=0.2
)

plt.xlabel("Training Set Size")
plt.ylabel("MSE")
plt.title("Decision Tree (Baseline) Learning Curve")
plt.legend()
plt.show()

In [ ]:
tree_baseline_results = {
    "Model": "Decision Tree (baseline)",
    "CV_R2": tree_cv_scores.mean(),
    "Validation_R2": tree_baseline_val_r2,
    "Validation_RMSE": tree_baseline_val_rmse,
    "Validation_MAE": tree_baseline_val_mae
}

tree_baseline_results

polynomyal

In [ ]:
poly_features = ["CPU_Frequency (GHz)", "RAM (GB)", "Weight (kg)", "ppi", "total_storage_gb"]

X_train_poly = X_train_main[poly_features].copy()

poly_model = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("model", LinearRegression())
])

poly_cv_scores = cross_val_score(
    poly_model,
    X_train_poly,
    y_train_main,
    cv=kf,
    scoring="r2"
)

print("Polynomial Regression CV R² scores:", poly_cv_scores)
print("Mean CV R²:", poly_cv_scores.mean())
print("Std CV R²:", poly_cv_scores.std())

In [ ]:
X_val_poly = X_val[poly_features].copy()

poly_model.fit(X_train_poly, y_train_main)

y_val_pred_poly = poly_model.predict(X_val_poly)

poly_val_r2 = r2_score(y_val, y_val_pred_poly)
poly_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_poly))
poly_val_mae = mean_absolute_error(y_val, y_val_pred_poly)

print("Polynomial Regression Validation R²:", poly_val_r2)
print("Polynomial Regression Validation RMSE:", poly_val_rmse)
print("Polynomial Regression Validation MAE:", poly_val_mae)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_val, y_val_pred_poly, alpha=0.7)
plt.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()])
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Polynomial Regression: Actual vs Predicted on Validation Set")
plt.show()

ridge tuning

In [ ]:
ridge_pipeline = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", Ridge())
])

ridge_param_grid = {
    "model__alpha": [0.001, 0.01, 0.1, 1, 5, 10, 20, 50, 100]
}

ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=kf,
    scoring="r2",
    n_jobs=-1
)

ridge_grid.fit(X_train_main, y_train_main)

print("Best Ridge alpha:", ridge_grid.best_params_)
print("Best Ridge CV R²:", ridge_grid.best_score_)

In [ ]:
best_ridge_model_tuned = ridge_grid.best_estimator_

best_ridge_model_tuned.fit(X_train_main, y_train_main)

y_val_pred_ridge_tuned = best_ridge_model_tuned.predict(X_val)

ridge_tuned_val_r2 = r2_score(y_val, y_val_pred_ridge_tuned)
ridge_tuned_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_ridge_tuned))
ridge_tuned_val_mae = mean_absolute_error(y_val, y_val_pred_ridge_tuned)

print("Tuned Ridge Validation R²:", ridge_tuned_val_r2)
print("Tuned Ridge Validation RMSE:", ridge_tuned_val_rmse)
print("Tuned Ridge Validation MAE:", ridge_tuned_val_mae)

ridge log1p

In [ ]:
best_ridge_log_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", Ridge(alpha=0.1))
])

best_ridge_log_model.fit(X_train_main, y_train_main_log)

y_val_pred_log = best_ridge_log_model.predict(X_val)
y_val_pred_log_back = np.expm1(y_val_pred_log)
y_val_pred_log_back = np.maximum(y_val_pred_log_back, 0)

ridge_log_val_r2 = r2_score(y_val, y_val_pred_log_back)
ridge_log_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_log_back))
ridge_log_val_mae = mean_absolute_error(y_val, y_val_pred_log_back)

print("Log-Ridge Validation R²:", ridge_log_val_r2)
print("Log-Ridge Validation RMSE:", ridge_log_val_rmse)
print("Log-Ridge Validation MAE:", ridge_log_val_mae)

tree tuning

In [ ]:
from scipy.stats import randint

tree_pipeline = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))
])

tree_param_dist = {
    "model__max_depth": randint(3, 25),
    "model__min_samples_split": randint(2, 20),
    "model__min_samples_leaf": randint(1, 10),
    "model__max_features": [None, "sqrt", "log2"]
}

tree_random = RandomizedSearchCV(
    estimator=tree_pipeline,
    param_distributions=tree_param_dist,
    n_iter=30,
    cv=kf,
    scoring="r2",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

tree_random.fit(X_train_main, y_train_main)

print("Best Tree params:", tree_random.best_params_)
print("Best Tree CV R²:", tree_random.best_score_)

In [ ]:
best_tree_model_tuned = tree_random.best_estimator_

best_tree_model_tuned.fit(X_train_main, y_train_main)

y_val_pred_tree_tuned = best_tree_model_tuned.predict(X_val)

tree_tuned_val_r2 = r2_score(y_val, y_val_pred_tree_tuned)
tree_tuned_val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_tree_tuned))
tree_tuned_val_mae = mean_absolute_error(y_val, y_val_pred_tree_tuned)

print("Tuned Tree Validation R²:", tree_tuned_val_r2)
print("Tuned Tree Validation RMSE:", tree_tuned_val_rmse)
print("Tuned Tree Validation MAE:", tree_tuned_val_mae)

comparison

In [ ]:
final_results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge (baseline)",
        "Lasso",
        "Decision Tree (baseline)",
        "Polynomial Regression",
        "Ridge (tuned)",
        "Decision Tree (tuned)",
        "Ridge (log target)"
    ],
    "CV_R2": [
        linear_cv_scores.mean(),
        ridge_cv_scores.mean(),
        lasso_cv_scores.mean(),
        tree_cv_scores.mean(),
        poly_cv_scores.mean(),
        ridge_grid.best_score_,
        tree_random.best_score_,
        np.nan
    ],
    "Validation_R2": [
        linear_val_r2,
        ridge_baseline_val_r2,
        lasso_val_r2,
        tree_baseline_val_r2,
        poly_val_r2,
        ridge_tuned_val_r2,
        tree_tuned_val_r2,
        ridge_log_val_r2
    ],
    "Validation_RMSE": [
        linear_val_rmse,
        ridge_baseline_val_rmse,
        lasso_val_rmse,
        tree_baseline_val_rmse,
        poly_val_rmse,
        ridge_tuned_val_rmse,
        tree_tuned_val_rmse,
        ridge_log_val_rmse
    ],
    "Validation_MAE": [
        linear_val_mae,
        ridge_baseline_val_mae,
        lasso_val_mae,
        tree_baseline_val_mae,
        poly_val_mae,
        ridge_tuned_val_mae,
        tree_tuned_val_mae,
        ridge_log_val_mae
    ]
})

final_results = final_results.sort_values(by="Validation_R2", ascending=False)
final_results

# **Create Sub File**

In [ ]:
final_ridge_model = Pipeline([
    ("preprocessor", linear_preprocessor),
    ("model", Ridge(alpha=0.1))
])

final_ridge_model.fit(X, y)

test_predictions = final_ridge_model.predict(test_features)

print("Prediction shape:", test_predictions.shape)
print("First 10 predictions:")
print(test_predictions[:10])

In [ ]:
submission = pd.DataFrame({
    "id": test_ids,
    "Price": test_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("\nSubmission shape:", submission.shape)